# Part I - Delay Cause Analysis in U.S. Airline On-Time Performance
## by Randall Sutton

## Introduction

This analysis explores the **Reporting Carrier On-Time Performance** dataset, which contains U.S. domestic flight records from 1987 to 2022 (2 million sampled rows). We focus on the **2005-2020** period, when delay cause data is consistently available.

**Primary question: What are the primary causes of flight delays, and do they differ by airline or region?**

The data contains five categories of delay:
- Carrier Delay
- Weather Delay
- NAS Delay
- Security Delay
- Late Aircraft Delay

## Preliminary Wrangling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

%matplotlib inline

In [ ]:
airline_data = pd.read_csv("data-visualization/airline_2m.csv", encoding="latin-1", low_memory=False)
print(f"Dataset shape: {airline_data.shape}")
airline_data.head()

In [ ]:
# Select columns relevant to delay cause analysis
cols = ['Year', 'Month', 'DayOfWeek', 'Reporting_Airline',
        'Origin', 'OriginCityName', 'OriginStateName',
        'Dest', 'DestCityName', 'DestStateName',
        'DepDelay', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15',
        'Cancelled', 'Diverted', 'Distance', 'DistanceGroup',
        'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

df = airline_data[cols].copy()

# Filter to 2005-2020 for consistency with delay cause data
df = df[(df['Year'] >= 2005) & (df['Year'] <= 2020)]

# Define delay cause columns
delay_causes = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

# Subset: flights with delay cause data (all 5 causes non-null)
delayed = df.dropna(subset=delay_causes).copy()

print(f"Total flights (2005-2020): {len(df):,}")
print(f"Flights with delay cause data: {len(delayed):,}")
print(f"Percentage with cause data: {len(delayed)/len(df)*100:.1f}%")
print(f"\nYear range: {df['Year'].min()} - {df['Year'].max()}")

In [ ]:
delayed[delay_causes].describe().round(1)

### What is the structure of your dataset?

The dataset contains 2 million sampled flight records with 109 columns covering flight timing, airline information, origin/destination airports, delay metrics, and delay causes. Each row represents a single domestic U.S. flight.

### What is/are the main feature(s) of interest in your dataset?

The five delay cause variables (`CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`) are the main features of interest. These record the number of delay minutes attributed to each cause for delayed flights.

### What features in the dataset do you think will help support your investigation into your feature(s) of interest?

- `Reporting_Airline` — to compare delay causes across carriers
- `OriginStateName` — to examine regional patterns
- `Month` and `Year` — to explore seasonal and long-term trends
- `ArrDel15` — binary delay indicator for computing delay rates
- `ArrDelay` — overall delay magnitude
- `Distance` — to explore relationship between flight length and delays

## Univariate Exploration

We start by examining the distributions of individual variables related to flight delays and delay causes.

### Q1: What is the overall distribution of arrival delays?
*(Histogram)*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
clipped = df['ArrDelay'].dropna().clip(-60, 60)
ax.hist(clipped, bins=80, edgecolor='none', alpha=0.7, color='steelblue')
ax.axvline(0, color='red', linestyle='--', alpha=0.6, label='On time')
ax.axvline(15, color='orange', linestyle='--', alpha=0.6, label='15-min threshold')
ax.set_xlabel('Arrival Delay (minutes)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Arrival Delays (clipped to -60 to 60 min)')
ax.legend()
plt.tight_layout();

> **Observation:** The distribution is right-skewed. Most flights arrive early (median = -4 min), and about 20% are delayed 15+ minutes.

### Q2: How are the individual delay causes distributed?
*(Histogram)*

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = sb.color_palette('Set2', 5)
for i, cause in enumerate(delay_causes):
    data = delayed[cause]
    data = data[data > 0].clip(0, 60)  # Exclude zeros
    axes[i].hist(data, bins=50, edgecolor='none', alpha=0.8, color=colors[i])
    axes[i].set_title(cause)
    axes[i].set_xlabel('Minutes')
    axes[i].set_ylabel('Frequency')
axes[5].set_visible(False)
plt.suptitle('Distribution of Delay Causes (non-zero delays only, clipped to 60 min)', fontsize=13)
plt.tight_layout()
plt.subplots_adjust(top=0.9);

> **Observation:** All causes are heavily right-skewed with large spikes at zero. SecurityDelay (99.6% zero) and WeatherDelay (93.9% zero) are rare. CarrierDelay, LateAircraftDelay, and NASDelay are the most common non-zero causes.

### Q3: What proportion of flights are delayed 15+ minutes?
*(Bar chart)*

In [ ]:
delay_counts = df['ArrDel15'].value_counts().sort_index()
labels = ['On Time / < 15 min', 'Delayed 15+ min']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, delay_counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white')
for i, v in enumerate(delay_counts.values):
    ax.text(i, v + len(df)*0.005, f'{v:,}\n({v/delay_counts.sum()*100:.1f}%)', ha='center', fontsize=11)
ax.set_ylabel('Number of Flights')
ax.set_title('Flight Delay Status (15-minute threshold)')
ax.set_ylim(0, delay_counts.max() * 1.15)
plt.tight_layout();

> **Observation:** 80% of flights are on time or under 15 minutes late. About 1 in 5 flights is significantly delayed.

### Q4: Which airlines have the most flights in this dataset?
*(Count plot)*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_airlines = df['Reporting_Airline'].value_counts().head(15)
sb.countplot(data=df[df['Reporting_Airline'].isin(top_airlines.index)],
             y='Reporting_Airline', hue='Reporting_Airline',
             order=top_airlines.index,
             palette='muted', legend=False, ax=ax)
ax.set_xlabel('Number of Flights')
ax.set_ylabel('Airline Code')
ax.set_title('Top 15 Airlines by Flight Count')
plt.tight_layout();

> **Observation:** A few airlines dominate the dataset with over 100k flights each, while most have significantly fewer. The distribution is uneven, with the top 3 airlines accounting for a large share of total flights.

### Q5: Which delay cause contributes the most minutes on average?
*(Bar chart)*

In [ ]:
avg_delays = delayed[delay_causes].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
avg_delays.plot(kind='barh', ax=ax, color=sb.color_palette('Set2', 5))
ax.set_xlabel('Average Delay (minutes)')
ax.set_title('Average Minutes by Delay Cause')
plt.tight_layout();

> **Observation:** LateAircraftDelay is the largest contributor (22.6 min avg), followed by CarrierDelay (17.3 min) and NASDelay (15.2 min). Weather and Security are minor.

### Univariate Discussion

Most flights arrive early, but 20% are delayed 15+ minutes. Among delay causes, LateAircraftDelay, CarrierDelay, and NASDelay are the three dominant factors. Weather and Security delays are rare.

## Bivariate Exploration

Next we investigate relationships between pairs of variables, focusing on how delays relate to flight characteristics, airlines, and time.

### Q6: Is there a relationship between flight distance and arrival delay?
*(Scatter plot)*

In [ ]:
sample = delayed.dropna(subset=['Distance']).sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sample['Distance'], sample['ArrDelay'].clip(0, 60),
           alpha=0.15, s=10, color='steelblue')
ax.axhline(15, color='orange', linestyle='--', alpha=0.4)
ax.set_xlabel('Distance (miles)')
ax.set_ylabel('Arrival Delay (minutes, clipped)')
ax.set_title('Flight Distance vs. Arrival Delay (5,000 sampled delayed flights)')
plt.tight_layout();

> **Observation:** There is no meaningful correlation between distance and arrival delay (r = -0.009). Short and long flights experience similar delay patterns.

### Q7: How do arrival delay distributions compare across airlines?
*(Box plot)*

In [ ]:
top_10 = df['Reporting_Airline'].value_counts().head(10).index
box_data = df[df['Reporting_Airline'].isin(top_10)].dropna(subset=['ArrDelay'])

# Order airlines by median delay
order = (box_data.groupby('Reporting_Airline')['ArrDelay']
         .median().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(12, 6))
sb.boxplot(data=box_data, x='Reporting_Airline', y='ArrDelay',
           hue='Reporting_Airline', order=order, palette='Set2',
           showfliers=False, legend=False, ax=ax)
ax.axhline(0, color='red', linestyle='--', alpha=0.4)
ax.axhline(15, color='orange', linestyle='--', alpha=0.4)
ax.set_xlabel('Airline')
ax.set_ylabel('Arrival Delay (minutes)')
ax.set_title('Arrival Delay Distribution by Airline (Top 10, outliers hidden)')
plt.tight_layout();

> **Observation:** All airlines have a negative median delay (most flights arrive early). Some airlines have higher medians and wider spreads, indicating more variable performance, while others show tighter, more consistent on-time arrival.

### Q8: How do average delay causes compare across airlines?
*(Heatmap)*

In [ ]:
top_15 = df['Reporting_Airline'].value_counts().head(15).index
airline_cause_avg = (delayed[delayed['Reporting_Airline'].isin(top_15)]
                     .groupby('Reporting_Airline')[delay_causes]
                     .mean())

fig, ax = plt.subplots(figsize=(10, 7))
sb.heatmap(airline_cause_avg, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Average Delay Minutes by Cause and Airline')
ax.set_ylabel('Airline')
plt.tight_layout();

> **Observation:** Delay profiles differ across airlines. Some have the highest LateAircraftDelay, others have the highest CarrierDelay or NASDelay. SecurityDelay is near zero for all airlines.

### Q9: How do delay rates vary across airlines?
*(Clustered bar chart)*

In [ ]:
top_15 = df['Reporting_Airline'].value_counts().head(15).index
airline_causes = (delayed[delayed['Reporting_Airline'].isin(top_15)]
                  .groupby('Reporting_Airline')[delay_causes]
                  .mean())
airline_causes.columns = [c.replace('Delay', '') for c in airline_causes.columns]

fig, ax = plt.subplots(figsize=(12, 6))
airline_causes.plot(kind='bar', ax=ax, width=0.8)
ax.set_ylabel('Average Delay (minutes)')
ax.set_xlabel('Airline')
ax.set_title('Average Delay Minutes by Cause and Airline')
ax.legend(title='Cause', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout();

> **Observation:** LateAircraftDelay is the tallest bar for most airlines. Some airlines stand out with very high LateAircraft but low NAS delay, while others show a more even split between the three main causes.

### Q10: How do delays vary by month?
*(Line chart)*

In [ ]:
monthly_delay = df.groupby('Month')['ArrDel15'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 5))
monthly_delay.plot(kind='line', marker='o', ax=ax, color='#e74c3c', linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Delay Rate')
ax.set_title('Flight Delay Rate by Month')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.grid(True, alpha=0.3)
plt.tight_layout();

> **Observation:** Delays peak in June (24.1%) and December, and are lowest in September (15.3%). This reflects summer storms and holiday travel driving more delays.

### Bivariate Discussion

Distance does not affect delay severity. Delay cause profiles differ across airlines — some are dominated by LateAircraftDelay, others by CarrierDelay or NASDelay. Delays follow a seasonal pattern, peaking in summer and December.

## Multivariate Exploration

Finally, we combine three or more variables to gain deeper insights into delay cause patterns across airlines, regions, and time.

### Q11: How do delay cause profiles differ across the top airlines?
*(Facet plot)*

In [ ]:
top_6 = df['Reporting_Airline'].value_counts().head(6).index
facet_data = delayed[delayed['Reporting_Airline'].isin(top_6)].copy()

# Melt delay causes into long format for faceting
facet_melted = facet_data[['Reporting_Airline'] + delay_causes].melt(
    id_vars='Reporting_Airline', var_name='Cause', value_name='Minutes')
facet_melted['Cause'] = facet_melted['Cause'].str.replace('Delay', '')
facet_melted['Minutes'] = facet_melted['Minutes'].clip(0, 200)

g = sb.FacetGrid(facet_melted, col='Reporting_Airline', col_wrap=3,
                 height=4, sharey=True)
g.map_dataframe(sb.barplot, x='Cause', y='Minutes', hue='Cause',
                palette='Set2', errorbar=None, legend=False,
                order=['Carrier','Weather','NAS','Security','LateAircraft'])
g.set_titles('{col_name}')
g.set_xticklabels(rotation=45)
g.set_axis_labels('Delay Cause', 'Avg Delay (min)')
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle('Average Delay by Cause — Top 6 Airlines (Faceted)', fontsize=14);

> **Observation:** The faceted view confirms that delay cause profiles vary by airline. Some are dominated by LateAircraftDelay, others by CarrierDelay. NASDelay is fairly consistent across airlines.

### Q11b: How are delay causes correlated with each other?
*(Plot Matrix)*

In [ ]:
# Sample for performance, exclude zeros, and clip for readability
pairplot_data = delayed[delay_causes].sample(2000, random_state=42)
pairplot_data = pairplot_data[(pairplot_data > 0).any(axis=1)].clip(0, 60)

g = sb.pairplot(pairplot_data, diag_kind='hist',
                plot_kws={'alpha': 0.3, 's': 10},
                diag_kws={'bins': 30})
g.fig.suptitle('Pairwise Relationships Between Delay Causes (non-zero delays, 2,000 sampled)', y=1.02)
plt.tight_layout();

> **Observation:** Delay causes are weakly negatively correlated. When one cause is high, the others tend to be low — delays are typically attributed to one primary cause, not spread across all five.

### Q12: How do departure delay, arrival delay, dominant cause, and distance interact?
*(Scatter plot with multiple encodings)*

In [ ]:
# Sample delayed flights with positive delays and determine dominant delay cause
sample_d = delayed[delayed['ArrDelay'] > 0].sample(min(3000, len(delayed)), random_state=42).copy()
sample_d['DominantCause'] = sample_d[delay_causes].idxmax(axis=1)

# Clip for readability
sample_d['DepDelay_clip'] = sample_d['DepDelay'].clip(0, 60)
sample_d['ArrDelay_clip'] = sample_d['ArrDelay'].clip(0, 60)

fig, ax = plt.subplots(figsize=(11, 7))
scatter = sb.scatterplot(data=sample_d, x='DepDelay_clip', y='ArrDelay_clip',
                         hue='DominantCause', size='Distance',
                         sizes=(10, 200), alpha=0.4, palette='Set2', ax=ax)
ax.set_xlabel('Departure Delay (minutes)')
ax.set_ylabel('Arrival Delay (minutes)')
ax.set_title('Departure vs Arrival Delay — colored by Dominant Cause, sized by Distance')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=1.5)
plt.tight_layout();

> **Observation:** Departure and arrival delays are strongly correlated. LateAircraftDelay and NASDelay are the most common dominant causes. Distance (point size) shows no clear pattern with delay severity.

### Q13: How do delay cause proportions compare across airlines?
*(Stacked bar chart)*

In [ ]:
top_15 = df['Reporting_Airline'].value_counts().head(15).index
airline_cause_totals = (delayed[delayed['Reporting_Airline'].isin(top_15)]
                        .groupby('Reporting_Airline')[delay_causes]
                        .sum())
airline_cause_pct = airline_cause_totals.div(airline_cause_totals.sum(axis=1), axis=0)
airline_cause_pct.columns = [c.replace('Delay', '') for c in airline_cause_pct.columns]

fig, ax = plt.subplots(figsize=(12, 6))
airline_cause_pct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', width=0.8)
ax.set_ylabel('Proportion of Total Delay Minutes')
ax.set_xlabel('Airline')
ax.set_title('Delay Cause Proportions by Airline (share of total delay minutes)')
ax.legend(title='Cause', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout();

> **Observation:** The proportion of delay minutes attributed to each cause varies by airline. Some airlines attribute over 50% of delay minutes to LateAircraftDelay, while others have a larger NAS or Carrier share.

### Q14: How do delay causes vary by origin state?
*(Heatmap)*

In [ ]:
top_20_states = df['OriginStateName'].value_counts().head(20).index
state_cause_avg = (delayed[delayed['OriginStateName'].isin(top_20_states)]
                   .groupby('OriginStateName')[delay_causes]
                   .mean())

fig, ax = plt.subplots(figsize=(10, 10))
sb.heatmap(state_cause_avg.sort_values('LateAircraftDelay', ascending=False),
           annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Average Delay Minutes by Cause and Origin State')
ax.set_ylabel('State')
plt.tight_layout();

> **Observation:** New Jersey has the highest LateAircraftDelay (27.2 min), likely due to Newark congestion. Ohio and Virginia lead in NASDelay. Michigan and Georgia have the highest WeatherDelay (4.2 min each).

### Q15: How have delay cause proportions changed over time?
*(Stacked area chart)*

In [ ]:
yearly_causes = delayed.groupby('Year')[delay_causes].mean()
yearly_causes.columns = [c.replace('Delay', '') for c in yearly_causes.columns]

fig, ax = plt.subplots(figsize=(14, 6))
yearly_causes.plot(kind='area', stacked=True, ax=ax, alpha=0.7, colormap='Set2')
ax.set_xlabel('Year')
ax.set_ylabel('Average Delay (minutes)')
ax.set_title('Delay Cause Trends Over Time (stacked area)')
ax.legend(title='Cause', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout();

> **Observation:** Total average delay minutes increased from 52.5 in 2005 to 69.9 in 2019, a 33% rise. LateAircraftDelay grew the most. 2020 shows a dip due to reduced travel from COVID-19.

### Multivariate Discussion

Delay cause profiles differ significantly across airlines. The pairplot confirmed that delay causes tend to be mutually exclusive — delays are usually attributed to one primary cause. Overall delays have worsened over time, driven by growth in LateAircraftDelay.

## Conclusions

1. **LateAircraftDelay is the primary cause of delays** (22.6 min avg), followed by CarrierDelay (17.3 min) and NASDelay (15.2 min). Weather and Security are minor.

2. **Delay cause profiles differ by airline.** Some airlines are dominated by late aircraft delays, others by carrier or NAS issues.

3. **Regional differences exist.** Hub-heavy states show higher LateAircraftDelay, while states with major ATC centers have higher NASDelay.

4. **Delays are seasonal and worsening.** June and December are the worst months. Overall delay severity increased 33% from 2005 to 2019, driven by LateAircraftDelay.